## Query Decomposition

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

C:\Users\kanha\AppData\Local\Temp\ipykernel_1028\3387885773.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
## Step 1:vectorstore and Load and split

loader=TextLoader("langchain_crewai_dataset.txt")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)
### Step 2:vectorstore
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
vectorstore=FAISS.from_documents(chunks,embeddings)
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":4,"lambda_mult":0.7})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
## Step 4: LLM and prompt
import os
from dotenv import load_dotenv
load_dotenv()
API=os.getenv("OPENAI_API_KEY")
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key=API)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A0B8899C90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A0B8BA9210>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
query_n_Prompt=PromptTemplate.from_template("""
You are a helpful asisstant . Decompose the following query to 2 to 4 smaller sub-question for this.
question:"{question}"
Sub-questions;                                                 
                                                
""")
query_decomposition_chain=query_n_Prompt|llm|StrOutputParser()
query_decomposition_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are a helpful asisstant . Decompose the following query to 2 to 4 smaller sub-question for this.\nquestion:"{question}"\nSub-questions;                                                 \n\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A0B8899C90>, async_client=<groq.resources.chat.completions.AsyncCompleti

In [6]:
qna=query_decomposition_chain.invoke({"question":"langchain-memory"})

In [7]:
print(qna)

To better understand the concept of "langchain-memory", I'll break it down into smaller sub-questions. Here are 4 sub-questions:

1. **What is Langchain?**: What is the definition and purpose of Langchain, and how does it relate to artificial intelligence or natural language processing?
2. **What is the role of memory in Langchain?**: How does memory function within the context of Langchain, and what type of information is stored in this memory?
3. **How does Langchain utilize memory to generate responses?**: What mechanisms or algorithms does Langchain use to access, retrieve, and generate responses based on the information stored in its memory?
4. **What are the limitations and potential applications of Langchain-memory?**: What are the current limitations and potential use cases of Langchain's memory capabilities, and how might they be improved or expanded upon in the future?


In [12]:
answer_prmpt=PromptTemplate.from_template("""
Answer the question based on the context below:
                                          
Context:
{context}
Question:{input}                         
                                                           """)
document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prmpt)

# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = query_decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = document_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)


In [13]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To better understand the comparison between LangChain and CrewAI in terms of memory and agents, let's break down the query into smaller sub-questions:
A: To better understand the comparison between LangChain and CrewAI in terms of memory and agents, let's break down the query into smaller sub-questions:

1. **How do LangChain agents operate in terms of memory and decision-making?**
   - According to the context, LangChain agents operate using a planner-executor model. This model allows for dynamic decision-making, branching logic, and context-aware memory use across steps. This implies that LangChain agents are capable of complex operations that involve adapting to new information and making decisions based on the context of the task at hand.

2. **What is the role of CrewAI in a hybrid system with LangChain?**
   - CrewAI manages role-based collaboration in a hybrid system where it is integrated with LangChain agents and tools. This suggests that while LangChain fo